In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import linregress, pearsonr

# ==============================================================================
# ПАРАМЕТРИ ВАРІАНТА 6
# Місто: Чернігів
# ==============================================================================
VARIANT = 6
CITY = "Чернігів"
MONTHLY_TEMP = [-5, -4, 1, 9, 15, 18, 20, 19, 13, 7, 1, -3]
MONTHS = ["Січ", "Лют", "Бер", "Кві", "Тра", "Чер", "Лип", "Сер", "Вер", "Жов", "Лис", "Гру"]

np.random.seed(VARIANT)

# Генерація вихідного набору даних
rows = []
for month, temp in zip(MONTHS, MONTHLY_TEMP):
    heating_days = np.clip(18 - temp + np.random.normal(0, 1.5), 0, 30)
    heating_days = round(heating_days)
    cost = 0.15 * heating_days + np.random.normal(0, 1.0)
    cost = round(max(cost, 0), 1)
    rows.append({
        "місто": CITY,
        "місяць": month,
        "температура": temp,
        "опалювальні_дні": heating_days,
        "витрати_на_опалення": cost,
    })

climate = pd.DataFrame(rows)

print(f"=== ПРАКТИКА 17: Варіант {VARIANT} ({CITY}) ===")
print(climate.to_string(index=False))

# ------------------------------------------------------------------------------
# ЗАВДАННЯ 1. Побудова регресійної моделі
# ------------------------------------------------------------------------------
print("\n=== Завдання 1: Побудова моделі регресії ===")

x = climate["температура"]
y = climate["витрати_на_опалення"]

res = linregress(x, y)
b0 = res.intercept
b1 = res.slope

print(f"Рівняння регресії: витрати ≈ {b0:.3f} + ({b1:.3f}) * температура")
print(f"Перетин (intercept, b0): {b0:.3f}")
print(f"Нахил (slope, b1):        {b1:.3f}")
print(f"p-value для нахилу:      {res.pvalue:.4e}")

task1_text = f"""
[Інтерпретація нахилу b1]
Нахил b1 = {b1:.3f} тис. грн/°C означає, що при підвищенні середньомісячної температури на 1°C
витрати на опалення зменшуються в середньому на {abs(b1):.3f} тис. грн (близько {abs(b1)*1000:.0f} грн).
Знак нахилу від'ємний (негативний), що є абсолютно логічним і відповідає фізичній природі процесу:
у тепліші місяці потреба в опаленні знижується, а в холодніші — зростає.
"""
print(task1_text)

# ------------------------------------------------------------------------------
# ЗАВДАННЯ 2. Коефіцієнт детермінації R²
# ------------------------------------------------------------------------------
print("=== Завдання 2: Коефіцієнт детермінації R² ===")

y_pred = b0 + b1 * x
residuals = y - y_pred

# Спосіб 1: Через квадрат rvalue
r2_method1 = res.rvalue ** 2

# Спосіб 2: Прямий розрахунок через суми квадратів (SS)
ss_res = np.sum((y - y_pred) ** 2)
ss_tot = np.sum((y - np.mean(y)) ** 2)
r2_method2 = 1 - (ss_res / ss_tot)

# Порівняння з r^2 з Практики 16
r_pearson, _ = pearsonr(x, y)
r2_pearson = r_pearson ** 2

print(f"R² (через res.rvalue**2):              {r2_method1:.5f}")
print(f"R² (через 1 - SS_res/SS_tot):          {r2_method2:.5f}")
print(f"r² (квадрат кореляції Пірсона r²):     {r2_pearson:.5f}")

task2_text = f"""
[Інтерпретація R²]
Отримане значення R² ≈ {r2_method1:.3f} (або {r2_method1*100:.1f}%).
Це означає, що {r2_method1*100:.1f}% загальної варіації (розкиду) витрат на опалення пояснюється
моделлю залежності від температури. Решта {(1 - r2_method1)*100:.1f}% розкиду залишається непоясненою
і зумовлена іншими факторами (випадковим шумом, коливаннями кількості опалювальних днів тощо).
Усі три способи обчислення дали точний збіг результатів.
"""
print(task2_text)

# ------------------------------------------------------------------------------
# ЗАВДАННЯ 3. Графік залишків та оцінка адекватності
# ------------------------------------------------------------------------------
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Лінійка регресії
ax1.scatter(x, y, color="blue", label="Фактичні дані", s=60)
ax1.plot(x, y_pred, color="red", label=f"Регресія: Y = {b0:.2f} + ({b1:.2f})X")
ax1.set_xlabel("Температура, °C")
ax1.set_ylabel("Витрати на опалення, тис. грн")
ax1.set_title("Лінійна регресія")
ax1.legend()
ax1.grid(True, linestyle="--", alpha=0.6)

# Графік залишків
ax2.scatter(x, residuals, color="purple", s=60)
ax2.axhline(0, color="gray", linestyle="--")
ax2.set_xlabel("Температура, °C")
ax2.set_ylabel("Залишки (Y_fact - Y_pred)")
ax2.set_title("Графік залишків")
ax2.grid(True, linestyle="--", alpha=0.6)

plt.tight_layout()
plt.savefig("regression_residuals_v6.png")
plt.close()

print("=== Завдання 3: Графік залишків ===")
print("Графік збережено як 'regression_residuals_v6.png'.")

task3_text = """
[Опис графіка залишків]
На графіку залишків спостерігається систематична нелінійна структура у зоні високих температур
(влітку, при X > 15°C):
Завдяки обмеженню max(cost, 0) у коді генерації, фактичні витрати влітку дорівнюють 0.0.
Проте лінійна модель передбачає від'ємні значення витрат при високих температурах.
В результаті залишки (Y - Y_pred) стають суто додатними і формують "вигин" угору в теплі місяці.
Це ілюструє важливе застереження: навіть високий R² (> 0.90) не гарантує повної адекватності
лінійної моделі, оскільки на межах діапазону виникають штучні нелінійні ефекти (плато на рівні 0).
"""
print(task3_text)

# ------------------------------------------------------------------------------
# ЗАВДАННЯ 4. Прогноз і застереження щодо екстраполяції
# ------------------------------------------------------------------------------
print("=== Завдання 4: Прогноз та екстраполяція ===")

x_new = -10.0
y_pred_new = b0 + b1 * x_new
x_min, x_max = x.min(), x.max()

print(f"Фактичний спостережений діапазон температур: [{x_min}°C; {x_max}°C]")
print(f"Точка для прогнозу: X = {x_new}°C")
print(f"Прогнозовані витрати при X = -10°C: {y_pred_new:.3f} тис. грн")

task4_text = f"""
[Оцінка прогнозу та екстраполяції]
Значення X = -10°C знаходиться поза спостереженим діапазоном [{x_min}°C; {x_max}°C] (X_new < X_min).
Отже, цей розрахунок є ЕКСТРАПОЛЯЦІЄЮ за межі даних.
Надійність такого прогнозу значно нижча з наступних причин:
1. Немає гарантії, що лінійний характер залежності зберігається при екстремально низьких температурах.
2. Можуть діяти технологічні обмеження (наприклад, максимальна потужність котелень чи насичення кількості опалювальних днів до 30–31 доби на місяць).
"""
print(task4_text)

# ------------------------------------------------------------------------------
# ЗАВДАННЯ 5. Порівняння з коефіцієнтом кореляції
# ------------------------------------------------------------------------------
print("=== Завдання 5: Порівняння p-value ===")

p_linreg = res.pvalue
p_pearson = r_pvalue = pearsonr(x, y).pvalue

print(f"p-value з linregress(): {p_linreg:.4e}")
print(f"p-value з pearsonr():   {p_pearson:.4e}")

task5_text = f"""
[Пояснення збігу p-value]
Значення p-value для linregress() ({p_linreg:.4e}) та pearsonr() ({p_pearson:.4e}) ідентичні.
Це пояснюється тим, що для простої лінійної регресії з одним предиктором нульові гіпотези:
- H0 (лінійна регресія): нахил b1 = 0
- H0 (кореляція Пірсона): коефіцієнт кореляції r = 0
є МА ТЕМАТИЧНО ЕКВІВАЛЕНТНИМИ. В обох випадках використовується одна і та сама t-статистика
t = r * sqrt(n - 2) / sqrt(1 - r^2) з однаковою кількістю степенів свободы df = n - 2.
"""
print(task5_text)

=== ПРАКТИКА 17: Варіант 6 (Чернігів) ===
   місто місяць  температура  опалювальні_дні  витрати_на_опалення
Чернігів    Січ           -5               23                  4.2
Чернігів    Лют           -4               22                  2.4
Чернігів    Бер            1               13                  2.9
Чернігів    Кві            9               11                  0.1
Чернігів    Тра           15                5                  0.3
Чернігів    Чер           18                4                  1.2
Чернігів    Лип           20                0                  1.2
Чернігів    Сер           19                0                  0.1
Чернігів    Вер           13                5                  0.6
Чернігів    Жов            7               12                  2.6
Чернігів    Лис            1               18                  4.5
Чернігів    Гру           -3               19                  2.4

=== Завдання 1: Побудова моделі регресії ===
Рівняння регресії: витрати ≈ 2.829 + (-0.

# Відповіді на контрольні питання
**Практична робота №17: Проста лінійна регресія: побудова моделі, коефіцієнти, залишки, коефіцієнт детермінації**
**Варіант 6:** м. Чернігів

---

### 1. Чи можете пояснити, чому метод найменших квадратів мінімізує саме суму квадратів запів, а не суму самих залишків чи їхніх модулів?

* **Чому не суму самих залишків $\sum (Y_i - \hat{Y}_i)$:**
  За визначенням оптимальної прямої, додатні та від'ємні залишки повністю взаємно нейтралізуються, тому їхня сума для будь-якої прямої, що проходить через точку середніх $(\bar{X}, \bar{Y})$, дорівнює нулю ($\sum e_i = 0$).

* **Чому піднесення до квадрата $\sum e_i^2$, а не модуль $\sum |e_i|$:**
  1. *Математична зручність:* Функція $Q = \sum (Y_i - b_0 - b_1 X_i)^2$ є гладкою та диференційовною на всьому просторі параметрів. Це дозволяє отримати точні аналітичні формули для $b_0$ та $b_1$ через взяття частинних похідних та прирівнювання їх до нуля. Функція модуля не є диференційовною в точці 0.
  2. *Штраф за великі помилки:* Піднесення до квадрата надає більшої ваги великим відхиленням, що мінімізує ризик появи ізольованих точок з грубими помилками прогнозу.
  3. *Статистичні властивості:* Відповідно до теореми Гаусса — Маркова, оцінки МНК є найліпшими лінійними незміщеними оцінками (BLUE) за умов виконання класичних передумов регресійного аналізу.

---

### 2. Що означає коефіцієнт нахилу $b_1$ змістовно для вашої пари змінних, і чим ця інтерпретація відрізняється від інтерпретації коефіцієнта кореляції $r$ для тієї самої пари?

* **Змістовне значення $b_1$:**
  Показує **кількісну абсолютну зміну** залежної змінної $Y$ (витрати на опалення у тис. грн) при зміні незалежної змінної $X$ (температура) на 1 одиницю вимірювання (на $1\text{ °C}$). Нахил має фізичну розмірність (тис. грн / °C).
* **Відмінність від коефіцієнта кореляції $r$:**
  * $r$ — це **безрозмірна величина**, що вимірює *силу та напрямок лінійного зв'язку* у діапазоні $[-1; 1]$. Вона симетрична відносно $X$ та $Y$.
  * $b_1$ — це **параметр спрямованої причинно-наслідкової або функціональної моделі**, що визначає конкретний фізичний масштаб зміни $Y$ залежно від $X$.

---

### 3. Чому високий $R^2$ сам по собі не є достатньою підставою вважати лінійну модель адекватною — що саме може показати графік залишків, чого не показує одне число $R^2$?

* **Обмеженість $R^2$:**
  $R^2$ описує лише частку поясненої дисперсії, але не перевіряє виконання фундаментальних припущень регресійного аналізу (лінійність, гетероскедастичність, відсутність автокореляції).
* **Що показує графік залишків:**
  1. *Приховану нелінійність:* Якщо реальний зв'язок є криволінійним (наприклад, квадратичним або U-подібним), $R^2$ може залишатися високим (наприклад, $0.90$), але на графіку залишків виникне виражена параболічна дуга.
  2. *Гетероскедастичність:* Зміну дисперсії залишків зі зростанням $X$ (наприклад, розширення «віяла» залишків).
  3. *Наявність аномальних точок та викидів.*

---

### 4. Що змінюється в надійності прогнозу, коли значення $x$, для якого робиться прогноз, виходить за межі спостереженого діапазону даних?

* **Перехід від інтерполяції до екстраполяції:**
  Прогноз усередині діапазону $[X_{\min}; X_{\max}]$ спирається на підтверджену експериментальними даними лінійну трендову лінію.
* **Причини падіння надійності при екстраполяції:**
  1. *Неперевіреність форми моделі:* Немає гарантії, що лінійний характер зберігається за межами вибірки (наприклад, витрати не можуть бути від'ємними при високих температурах або перевищувати максимальну потужність мереж при критичних морозах).
  2. *Розширення довірчого інтервалу:* Довірчі інтервали для прогнозу $\hat{Y}_0$ мають форму гіперболи $SE(\hat{Y}_0) \propto \sqrt{1 + \frac{1}{n} + \frac{(X_0 - \bar{X})^2}{\sum (X_i - \bar{X})^2}}$. Чим далі $X_0$ знаходиться від $X_{\bar{X}}$, тим ширшим і менш точним стає інтервал оцінки.